# Day 3 · 04 · SQL Joins

A join combines rows from two tables by matching a key. Choosing the join type decides **which rows survive when there is no match**.

**In this notebook**
- The retail schema and how the tables relate
- `INNER JOIN`
- `LEFT JOIN` and finding rows with no match
- Filtering in `ON` versus `WHERE`
- Joining several tables
- `RIGHT` and `FULL OUTER JOIN`
- Self joins and `CROSS JOIN`
- The fan-out pitfall and semi-joins with `EXISTS`

**Dataset:** `retail.db`, a SQLite database with the tables `customers`, `products`, `orders`, `order_items` and `staff`

Every query runs through a small helper that returns a DataFrame, so results display as tables.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA = Path("../datasets").resolve()

# Open the database read-only so nothing in this lesson can change it.
conn = sqlite3.connect(f"{(DATA / 'retail.db').as_uri()}?mode=ro", uri=True)


def sql(query, params=()):
    """Run a query and return the result as a DataFrame."""
    return pd.read_sql_query(query, conn, params=params)


print("SQLite", sqlite3.sqlite_version)

SQLite 3.50.4


## 1. The schema
| Table | One row is | Primary key | Foreign keys |
| --- | --- | --- | --- |
| `customers` | a customer | `customer_id` | |
| `products` | a product | `product_id` | |
| `staff` | an employee | `staff_id` | `manager_id` → `staff` |
| `orders` | an order | `order_id` | `customer_id` → `customers`, `staff_id` → `staff` (empty for online orders) |
| `order_items` | one product line of an order | `order_id` + `line_no` | `order_id` → `orders`, `product_id` → `products` |

```
customers 1 ──< orders 1 ──< order_items >── 1 products
                  │
staff 1 ──────────┘        staff.manager_id ──> staff.staff_id
```

A **primary key** identifies a row. A **foreign key** points at a primary key in another table. Joins follow those arrows.

In [2]:
tables = sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")["name"]

pd.DataFrame(
    {
        "table": tables,
        "rows": [conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables],
        "columns": [", ".join(row[1] for row in conn.execute(f"PRAGMA table_info({t})")) for t in tables],
    }
)

,table,rows,columns
0,customers,250,"customer_id, name, email, city, country, segme..."
1,order_items,4439,"order_id, line_no, product_id, quantity, unit_..."
2,orders,2000,"order_id, customer_id, order_date, status, cha..."
3,products,40,"product_id, product_name, category, unit_price..."
4,staff,24,"staff_id, name, role, department, manager_id, ..."


## 2. `INNER JOIN`: only rows that match on both sides
- `ON` states how the tables relate.
- Table **aliases** (`o`, `c`) keep the query short. Prefix every column with its alias so it is clear where it comes from.
- `JOIN` on its own means `INNER JOIN`.

In [3]:
sql("""
SELECT o.order_id, o.order_date, o.status, c.name AS customer, c.country
FROM orders AS o
JOIN customers AS c ON c.customer_id = o.customer_id
ORDER BY o.order_id
LIMIT 5
""")

,order_id,order_date,status,customer,country
0,10001,2024-01-01,completed,Chloe Brown,Spain
1,10002,2024-01-01,returned,Chloe Schmidt,India
2,10003,2024-01-01,completed,Sofia Miller,UK
3,10004,2024-01-02,completed,Grace Fischer,India
4,10005,2024-01-03,completed,Felix Moreau,Spain


## 3. `LEFT JOIN`: keep every row from the left table
Rows from the left table with no match are kept, and the right table's columns are filled with `NULL`.

`COUNT(o.order_id)` counts only non-`NULL` values, so a customer with no orders correctly shows `0`. `COUNT(*)` would show `1`.

In [4]:
sql("""
SELECT c.customer_id, c.name, COUNT(o.order_id) AS orders
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY orders, c.customer_id
LIMIT 6
""")

,customer_id,name,orders
0,C0004,Lucia Murphy,0
1,C0009,Jonas Lopez,0
2,C0012,Grace Taylor,0
3,C0028,Lucas Wilson,0
4,C0034,Noah Tremblay,0
5,C0037,Rohan Iyer,0


### Anti-join: rows with no match
`LEFT JOIN ... WHERE right.key IS NULL` keeps exactly the left rows that found no partner. Here: customers who have never ordered.

In [5]:
sql("""
SELECT c.country, COUNT(*) AS customers_without_orders
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
GROUP BY c.country
ORDER BY customers_without_orders DESC
""")

,country,customers_without_orders
0,USA,9
1,Germany,9
2,India,8
3,France,8
4,UK,6
5,Spain,3
6,Canada,2


In [6]:
# The three numbers must reconcile: all customers = with orders + without orders.
sql("""
SELECT
    (SELECT COUNT(*) FROM customers) AS all_customers,
    (SELECT COUNT(DISTINCT customer_id) FROM orders) AS with_orders,
    (SELECT COUNT(*) FROM customers AS c
     LEFT JOIN orders AS o ON o.customer_id = c.customer_id
     WHERE o.order_id IS NULL) AS without_orders
""")

,all_customers,with_orders,without_orders
0,250,205,45


## 4. Pitfall: a filter in `WHERE` can undo a `LEFT JOIN`
- A condition in **`ON`** decides which right-hand rows *match*. Unmatched left rows are still kept.
- A condition in **`WHERE`** runs *after* the join. A test on a right-hand column rejects the `NULL` rows, which quietly turns the left join into an inner join.

Question: how many customers appear when we attach only their **returned** orders?

In [7]:
sql("""
SELECT 'filter in ON' AS version, COUNT(DISTINCT c.customer_id) AS customers_kept
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id AND o.status = 'returned'

UNION ALL

SELECT 'filter in WHERE', COUNT(DISTINCT c.customer_id)
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE o.status = 'returned'
""")

,version,customers_kept
0,filter in ON,250
1,filter in WHERE,64


## 5. Joining several tables
Chain joins one relationship at a time. Start from the table at the level of detail you want in the result. Here that is `order_items`, one row per order line.

In [8]:
sql("""
SELECT o.order_id,
       c.name AS customer,
       p.product_name,
       p.category,
       i.quantity,
       ROUND(i.quantity * i.unit_price * (1 - i.discount), 2) AS revenue
FROM order_items AS i
JOIN orders    AS o ON o.order_id    = i.order_id
JOIN customers AS c ON c.customer_id = o.customer_id
JOIN products  AS p ON p.product_id  = i.product_id
ORDER BY o.order_id, i.line_no
LIMIT 6
""")

,order_id,customer,product_name,category,quantity,revenue
0,10001,Chloe Brown,Cast Iron Pan,Home,1,38.0
1,10001,Chloe Brown,Pandas in Practice,Books,4,156.0
2,10002,Chloe Schmidt,USB-C Hub,Electronics,1,39.5
3,10002,Chloe Schmidt,Electric Kettle,Home,1,34.5
4,10003,Sofia Miller,Board Game Classic,Toys,1,28.0
5,10004,Grace Fischer,Resistance Bands,Sports,2,31.0


## 6. `RIGHT JOIN` and `FULL OUTER JOIN`
- `RIGHT JOIN` keeps every row of the **right** table. It is a `LEFT JOIN` with the tables swapped, and most people simply write the left version.
- `FULL OUTER JOIN` keeps unmatched rows from **both** sides. Use it to reconcile two lists.

SQLite supports both from version 3.39.

First, products that were never sold:

In [9]:
sql("""
SELECT p.product_id, p.product_name, p.category
FROM order_items AS i
RIGHT JOIN products AS p ON p.product_id = i.product_id
WHERE i.order_id IS NULL
""")

,product_id,product_name,category
0,P038,Science Kit,Toys
1,P039,Drawing Tablet Kids,Toys
2,P040,Kite,Toys


Now reconcile staff with orders. Three outcomes are possible:
- **matched**: an in-store order with its sales rep
- **order without staff**: online orders, which have no rep
- **staff without orders**: employees who do not take orders

In [10]:
sql("""
SELECT CASE
           WHEN s.staff_id IS NULL THEN 'order without staff'
           WHEN o.order_id IS NULL THEN 'staff without orders'
           ELSE 'matched'
       END AS outcome,
       COUNT(*) AS row_count
FROM staff AS s
FULL OUTER JOIN orders AS o ON o.staff_id = s.staff_id
GROUP BY outcome
ORDER BY row_count DESC
""")

,outcome,row_count
0,order without staff,1596
1,matched,404
2,staff without orders,16


## 7. Self join: a table joined to itself
`staff.manager_id` points at another row of `staff`. Give the table two aliases and treat them as two tables: `e` for the employee, `m` for the manager.

A `LEFT JOIN` keeps the chief executive, who has no manager.

In [11]:
sql("""
SELECT e.staff_id, e.name AS employee, e.role, m.name AS manager, m.role AS manager_role
FROM staff AS e
LEFT JOIN staff AS m ON m.staff_id = e.manager_id
ORDER BY e.staff_id
LIMIT 8
""")

,staff_id,employee,role,manager,manager_role
0,S01,Liam Menon,Chief Executive,NaN,NaN
1,S02,Lucas Schmidt,Head of Sales,Liam Menon,Chief Executive
2,S03,Diego Iyer,Head of Support,Liam Menon,Chief Executive
3,S04,Aarav Garcia,Head of Operations,Liam Menon,Chief Executive
4,S05,Emma Sharma,Sales Team Lead,Lucas Schmidt,Head of Sales
5,S06,Lea Weber,Sales Team Lead,Lucas Schmidt,Head of Sales
6,S07,Olivia Garcia,Sales Rep,Emma Sharma,Sales Team Lead
7,S08,Ananya Miller,Sales Rep,Emma Sharma,Sales Team Lead


## 8. `CROSS JOIN`: every combination
A cross join pairs every row of one table with every row of the other. It has no `ON`.

A useful pattern is to build a **complete grid** of categories, then left-join the facts onto it, so combinations with no data show `0` instead of going missing from a report.

In [12]:
sql("""
SELECT ch.channel, st.status, COUNT(o.order_id) AS orders
FROM (SELECT DISTINCT channel FROM orders) AS ch
CROSS JOIN (SELECT DISTINCT status FROM orders) AS st
LEFT JOIN orders AS o ON o.channel = ch.channel AND o.status = st.status
GROUP BY ch.channel, st.status
ORDER BY ch.channel, orders DESC
""")

,channel,status,orders
0,mobile,completed,504
1,mobile,shipped,56
2,mobile,returned,36
3,mobile,cancelled,30
4,store,completed,318
5,store,cancelled,35
6,store,shipped,31
7,store,returned,20
8,web,completed,759
9,web,cancelled,91


## 9. Pitfall: fan-out
Joining a "one" table to a "many" table **repeats** the "one" rows. An order with three lines appears three times, so counting or summing order-level values after the join gives inflated results.

Check the row count before and after every join. If it grew, you are now at the grain of the "many" table.

In [13]:
sql("""
SELECT (SELECT COUNT(*) FROM orders)  AS orders_in_table,
       COUNT(*)                       AS rows_after_join,
       COUNT(DISTINCT o.order_id)     AS distinct_orders
FROM orders AS o
JOIN order_items AS i ON i.order_id = o.order_id
""")

,orders_in_table,rows_after_join,distinct_orders
0,2000,4439,2000


## 10. Semi-join with `EXISTS`
"Customers who bought anything from Electronics" is a yes/no question per customer. `EXISTS` answers it without multiplying rows, and it stops at the first match.

The join version below needs `DISTINCT` to undo the fan-out. Both give the same number.

In [14]:
sql("""
SELECT
    (SELECT COUNT(*)
     FROM customers AS c
     WHERE EXISTS (SELECT 1
                   FROM orders AS o
                   JOIN order_items AS i ON i.order_id = o.order_id
                   JOIN products AS p ON p.product_id = i.product_id
                   WHERE o.customer_id = c.customer_id
                     AND p.category = 'Electronics')) AS with_exists,
    (SELECT COUNT(DISTINCT o.customer_id)
     FROM orders AS o
     JOIN order_items AS i ON i.order_id = o.order_id
     JOIN products AS p ON p.product_id = i.product_id
     WHERE p.category = 'Electronics') AS with_join_and_distinct
""")

,with_exists,with_join_and_distinct
0,164,164


In [15]:
conn.close()

## Summary
| Join | Keeps |
| --- | --- |
| `INNER JOIN` | rows with a match on both sides |
| `LEFT JOIN` | all left rows, `NULL` where the right has no match |
| `RIGHT JOIN` | all right rows |
| `FULL OUTER JOIN` | all rows from both sides |
| `CROSS JOIN` | every combination |
| Self join | a table matched against itself through two aliases |
| Anti-join | `LEFT JOIN ... WHERE right.key IS NULL`, or `NOT EXISTS` |
| Semi-join | `WHERE EXISTS (...)` |

## Key takeaways
- Know the grain of each table: what does one row represent?
- Decide which unmatched rows you need before choosing the join type.
- Put conditions on the optional side of a `LEFT JOIN` in `ON`, not `WHERE`.
- Compare row counts before and after a join to catch fan-out.

## Practice
1. List each sales rep with the number of in-store orders they handled, including reps with none.
2. Find products that were sold but never returned.
3. Show each employee with their manager's manager.
4. For every country and channel, count the orders, including combinations with zero.